# 天文影像描述 LoRA — Colab 訓練流程

從頭到尾跑一次：下載資料 → 切分 → 訓練 → 評估 → 上傳權重。

**執行階段 → 變更執行階段類型 → T4 GPU**，然後從上往下一格一格跑。

---

### 兩個刻意的設計，先看懂再跑

**1. 每一步都是獨立的子行程（`!python -m ...`）。**
所以某一步失敗時，它佔的顯存會隨行程結束**自動釋放**——不會像在單一 notebook 裡
那樣，失敗的模型被 traceback 抓著不放，下一次執行直接 OOM。這是刻意的。

**2. 切分（`src.data.split`）一定在訓練之前。**
訓練只讀 `data/splits/train.json`，評估只讀 `test.json`。原始 notebook 是先用全部
資料訓練、訓練完才切 train/test，於是測試集 100% 出現在訓練資料裡。不要調換順序。

---

### 只想訓練完直接部署？

**第 7 節（評估）整段可以跳過。** 最短路徑：

`0 → 1 → 2 → 3 → 4 → 5 → 6 → 8（權重上傳 HF）→ 9（部署 Space）`

評估不影響模型本身，只是產生成果表的數字。之後想補，隨時可以回來跑第 7 節。


## 0. 確認拿到 GPU


In [ ]:
!nvidia-smi


## 1. 取得專案

已經 clone 過就會自動改成 `git pull`。


In [ ]:
import os
import subprocess

REPO = "https://github.com/lee851104/lora_mars.git"
WORKDIR = "/content/lora_mars"

# os.chdir 而不是 %cd：純 Python，在 if/else 裡面不會出事，
# 而且後面每一格的 ! 指令都會繼承這個工作目錄。
if os.path.isdir(WORKDIR):
    os.chdir(WORKDIR)
    subprocess.run(['git', 'pull'], check=True)
else:
    subprocess.run(['git', 'clone', REPO, WORKDIR], check=True)
    os.chdir(WORKDIR)

print('cwd:', os.getcwd())
print(sorted(os.listdir()))


## 2. 安裝套件

**不要在 Colab 上跑 `uv sync`。** Colab 預裝的 torch 已經跟它的 CUDA 版本對好了，
重裝一次就會撞回 transformers / huggingface_hub 的版本衝突。

`make setup-colab` 刻意不碰 torch：unsloth 堆疊全部 `--no-deps` 安裝，
`transformers==4.57.3` 最後釘（它會順便把 `huggingface_hub` 拉回 `<1.0`，
這正是 `additional_chat_templates` 404 的解法）。

大約 2～3 分鐘。


In [ ]:
!make setup-colab


### 確認裝對了

`cuda True` 且 `transformers 4.57.3` 才往下走。

> 這格若報版本錯誤：**執行階段 → 重新啟動工作階段**，再從第 1 格重跑。
> pip 裝好的東西會留著，只有幾秒鐘的事。


In [ ]:
!python -c "import torch, transformers; print('torch', torch.__version__, '| transformers', transformers.__version__, '| cuda', torch.cuda.is_available())"


## 3. 下載資料集

250 張天文照片 + caption，還原成 `data/raw/{data.json, images/}`。

**注意這不是火星專屬資料集**：Earth 77 / Mars 54 / Mars Rover 46 /
Milky Way 45 / Hubble 28。火星相關佔 40%，最大單一類別其實是地球。


In [ ]:
!python -m src.data.download


## 4. 清洗 + 切分

清洗預設 `conservative`（只正規化空白），保留 `M31`、`Apollo 11` 這類專有名詞。
原專案的 `aggressive` 模式會把所有非字母字元刪光，那些資訊就永久消失。

切分產生 train / val / test（約 200 / 25 / 25），並寫出一個 `split_hash`。
訓練時這個 hash 會記進權重的 `train_meta.json`，之後偷改切分會被測試抓到。


In [ ]:
!python -m src.data.build
!python -m src.data.split


## 5. 冒煙測試（2 步）

**先跑這格，不要直接跑正式訓練。** 只跑 2 步，確認整條 GPU 路徑通。

特別盯 loss：**Gemma 3 用 bf16 訓練，而 T4 沒有 bf16 硬體**。unsloth 有專門處理
（activation 走 bf16/fp32、只有 matmul 降 fp16、layernorm 升 fp32），但這是所有
預設裡最容易出現 `nan` 的一個。

看到 `nan`：把 learning rate 再往下調（預設已是 1e-4），或換小模型：

```
!python -m src.models.train --override-file configs/models/qwen2_vl_2b.yaml train.max_steps=2
```


In [ ]:
!python -m src.models.train train.max_steps=2 train.warmup_steps=0


## 6. 正式訓練

預設 30 步、有效 batch 8，T4 上大約 10～20 分鐘。
權重寫到 `models/lora/`，連同 `train_meta.json`（記錄 split_hash 與訓練了哪些 id）。

> 想跑久一點：加 `train.max_steps=120`。但訓練樣本只有 200 筆，
> 步數拉太高很快就是在背答案。


In [ ]:
!python -m src.models.train


## 7. 評估（可跳過）

> 只想拿到權重去部署的話，這一節整段跳過，直接到第 8 節。

跑**兩次**：一次不掛 LoRA（對照組），一次掛上（實驗組）。
只有實驗組的分數等於沒有證據。

輸出檔名自動分開（`eval_test_base.json` vs `eval_test.json`），不會互相覆蓋。


### 7a. 設定 API key（LLM-as-judge 要用）

左側邊欄的 🔑 **密鑰** → 新增 `ANTHROPIC_API_KEY` → 打開「筆記本存取權」。

25 張圖用 `claude-opus-5` 大約 **$0.75**。

**沒有 key 也能跑**，只是少掉幻覺偵測——下一格會自動改成只跑 CLIPScore。


In [ ]:
import os

try:
    from google.colab import userdata
    os.environ['ANTHROPIC_API_KEY'] = userdata.get('ANTHROPIC_API_KEY')
    METRICS = 'eval.metrics=[clipscore,llm_judge]'
    print('有 API key：CLIPScore + LLM-as-judge')
except Exception as error:
    METRICS = 'eval.metrics=[clipscore]'
    print('沒有 API key（' + type(error).__name__ + '）：只跑 CLIPScore')

print(METRICS)


### 7b. 對照組：未微調的基礎模型


In [ ]:
!python -m src.models.evaluate eval.use_adapter=false "$METRICS"


### 7c. 實驗組：掛上 LoRA

> 這是另一個獨立行程，上一格的模型顯存已完全釋放，不會 OOM。


In [ ]:
!python -m src.models.evaluate "$METRICS"


### 7d. 兩邊擺一起看

**看信賴區間有沒有重疊，不要比小數點。** held-out 只有二十幾筆。

CLIPScore 一定要對照 `clipscore_reference`（人類描述在同一批圖上的分數＝天花板）。


In [ ]:
import json
from pathlib import Path

def load(name):
    path = Path('reports') / name
    return json.loads(path.read_text(encoding='utf-8')) if path.exists() else None

for label, name in (('BASE（未微調）', 'eval_test_base.json'),
                    ('LoRA（微調後）', 'eval_test.json')):
    report = load(name)
    if report is None:
        print(label + ': 還沒跑')
        continue
    clip = report['metrics'].get('clipscore', {})
    judge = report['metrics'].get('llm_judge', {})
    print('--- ' + label + '  n=' + str(report['n_samples']) + ' ---')
    if 'clipscore' in clip:
        print('  CLIPScore   ', clip['clipscore'], ' CI95', clip.get('clipscore_ci95'))
        print('  人類天花板  ', clip['clipscore_reference'])
    if judge.get('n_judged'):
        print('  judge 整體  ', judge['overall'], ' CI95', judge.get('overall_ci95'))
        print('  judge 正確  ', judge['accuracy'])
        print('  幻覺數/張   ', judge['hallucination_count'])
        print('  無幻覺比例  ', round(judge['hallucination_free_rate'] * 100), '%')
    print()


## 8. 上傳權重到你的 Hugging Face

Space 讀不到 Colab 上的檔案，所以要先上傳。

左側 🔑 新增 `HF_TOKEN`（<https://huggingface.co/settings/tokens>，要 **write** 權限）。

只會上傳 adapter（幾十 MB），不會上傳基礎模型。


In [ ]:
import os

from google.colab import userdata
os.environ['HF_TOKEN'] = userdata.get('HF_TOKEN')

# 改成你自己的
ADAPTER_REPO = 'lee851104/gemma3-4b-astronomy-lora'

!python -m src.models.upload upload.repo_id=$ADAPTER_REPO


## 9. 部署到 Hugging Face Space

`build_space --push` 會組出 Space 需要的檔案（`app.py`、`requirements.txt`、
帶 frontmatter 的 README、`src/`、`configs/`）並直接上傳，不用先抓回本機再 git push。

**部署完還有兩件事必須在 Space 網頁上做，API 設不了：**

1. **Settings → Hardware → T4 small 或更好。**
   免費的 CPU basic **完全載不動**這個模型，不是慢而是跑不起來。
2. **Settings → Variables → 新增 `LORA_REPO_ID`**，值就是第 8 節上傳的 adapter repo。
   沒設的話 Space 會跑原廠模型，介面會明講「找不到 LoRA 權重」。

> 不想付 GPU 費用的話，有個免費替代：在 Colab 上跑
> `!python -m src.serving.app serving.share=true`，會給你一個 72 小時有效的
> `*.gradio.live` 公開連結。展示夠用。


In [ ]:
import os

from google.colab import userdata
os.environ['HF_TOKEN'] = userdata.get('HF_TOKEN')

# 改成你自己的
SPACE_REPO = 'lee851104/astrovision-lora'

!python -m src.serving.build_space --push space.repo_id=$SPACE_REPO


## 10. 把報告帶回去 commit（有跑第 7 節才需要）

`reports/` 的 JSON 與圖表要進版控——README 的成果表需要有依據可查。
權重和資料不進版控（`.gitignore` 已排除）。

下載後在本機 repo 解壓到 `reports/`，再 commit push。

> 不建議在 Colab 上直接 `git push`：那需要把 GitHub token 放進 Colab。


In [ ]:
!zip -qr reports.zip reports/
!ls -la reports.zip

from google.colab import files
files.download('reports.zip')


---

## 出事的時候

| 症狀 | 處理 |
|---|---|
| loss 變 `nan` | Gemma 3 在 T4 的 fp16 問題。調低 `train.learning_rate`，或改用 `qwen2_vl_2b` |
| `CUDA out of memory` | **重新啟動工作階段**再跑，不要直接重跑同一格 |
| `PassManager::run failed` | Triton 在 T4 的已知問題：`!pip uninstall -y cut_cross_entropy` 後重跑 |
| `additional_chat_templates` 404 | transformers 沒釘到 4.57.3，重跑第 2 格 |
| judge 全部失敗 | 檢查 `ANTHROPIC_API_KEY` 有沒有打開「筆記本存取權」 |

換基礎模型：任何 `python -m ...` 後面加 `--override-file configs/models/<名稱>.yaml`。
可選 `gemma3_4b`（預設）、`qwen2_5_vl_7b`、`qwen2_vl_2b`、`llama3_2_11b_vision`。
**同一次實驗的 train / eval 要用同一個**，否則 adapter 對不上基礎模型。
